# NumCompute — End-to-End Quickstart Demo

**Assignment 2.1 · Programming Task 1 · NumPy toolkit**

This notebook demonstrates the team's existing `numcompute` package. It uses only Python and NumPy for the computing operations and runs without external datasets.

**Demo flow:** CSV loading → preprocessing and pipeline → sorting/ranking → statistics → metrics → finite-difference derivatives → reproducible benchmark.

*Run the notebook from the `NumCompute` project root or its `demo/` folder.*

## 1. Setup

Locate the package and import the existing modules. This handles running from either the project root or the `demo/` folder.

In [ ]:
import sys
from pathlib import Path

import numpy as np

# Locate the folder containing the numcompute package.
locations = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    (folder for base in locations
     for folder in (base, base / "NumCompute")
     if (folder / "numcompute" / "__init__.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open this notebook inside the NumCompute project folder.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from numcompute import benchmarking, io, metrics, optim, preprocessing, rank, sort_search, stats
from numcompute.pipeline import Pipeline

np.set_printoptions(precision=3, suppress=True)
print("NumCompute imported successfully.")
print("Project:", project_root.name)
print("NumPy version:", np.__version__)

## 2. Read CSV data (including missing values)

The example CSV has six rows and two blank cells. `io.load_csv` converts missing numeric values to `NaN`, which can then be handled by a preprocessor.

In [ ]:
import tempfile

csv_text = "1,10,100\n2,,110\n3,30,120\n4,40,\n5,50,140\n6,60,150\n"
with tempfile.TemporaryDirectory() as temporary_folder:
    csv_path = Path(temporary_folder) / "example.csv"
    csv_path.write_text(csv_text, encoding="utf-8")
    X = io.load_csv(csv_path, missing="fill", fill_value=np.nan)

print("Loaded data (rows, features):", X.shape)
print(X)
print("Missing cells:", int(np.isnan(X).sum()))
assert X.shape == (6, 3)
assert np.isnan(X).sum() == 2

## 3. Preprocess data and chain a pipeline

`SimpleImputer` replaces missing values; `StandardScaler` standardizes features; `MinMaxScaler` maps features to 0–1. `Pipeline` performs multiple steps with one call. We also show the separate `OneHotEncoder` for categories.

We also fit a pipeline on training rows, then reuse the fitted transforms on new rows.

In [ ]:
# Pipeline: replace NaN with 0, then standardize each numeric column.
pipe = Pipeline([
    ("impute", preprocessing.SimpleImputer(fill_value=0)),
    ("scale", preprocessing.StandardScaler()),
])
X_scaled = pipe.fit_transform(X)

X_imputed = preprocessing.SimpleImputer(fill_value=0).fit_transform(X)
X_minmax = preprocessing.MinMaxScaler().fit_transform(X_imputed)

print("Standardized data (first 3 rows):")
print(X_scaled[:3])
print("Feature means after scaling:", np.mean(X_scaled, axis=0))
print("MinMax output range:", X_minmax.min(), "to", X_minmax.max())

categories = np.array([["red"], ["blue"], ["red"], [np.nan]], dtype=object)
encoder = preprocessing.OneHotEncoder()
encoded = encoder.fit_transform(categories)
print("One-hot categories:", encoder.categories_[0])
print("One-hot encoded rows:\n", encoded)

np.testing.assert_allclose(np.mean(X_scaled, axis=0), np.zeros(3), atol=1e-12)
assert not np.isnan(X_scaled).any()

### 3.1 Apply a fitted pipeline to new data

Fit the imputer and scaler on training rows only. Then reuse the fitted parameters for new rows without refitting.

In [ ]:
# Fit on training data and apply the same transforms to unseen data.
X_train = X[:4]
X_new = np.array([[7.0, np.nan, 160.0], [8.0, 80.0, 170.0]])

new_pipe = Pipeline([
    ("impute", preprocessing.SimpleImputer(fill_value=0)),
    ("scale", preprocessing.StandardScaler()),
])
new_pipe.fit(X_train)
new_scaled = new_pipe.transform(X_new)

# Compare with a calculation using only the training data parameters.
train_imputed = np.nan_to_num(X_train, nan=0.0)
expected_new = (np.nan_to_num(X_new, nan=0.0) - train_imputed.mean(axis=0)) / train_imputed.std(axis=0)
np.testing.assert_allclose(new_scaled, expected_new, atol=1e-12)
print("New data transformed with training parameters:\n", new_scaled)
print("Pipeline fit/transform check passed")

### 3.2 Integration check: CSV text → OneHotEncoder

The earlier category example uses an in-memory array. This additional example checks that categorical values loaded through `io.load_csv(dtype=object)` can be passed directly into `OneHotEncoder`. It also validates the expected output.

In [ ]:
# Read a CSV containing text and numeric columns.
category_csv = "red,1\nblue,2\nred,3\n"
with tempfile.TemporaryDirectory() as temp_folder:
    category_path = Path(temp_folder) / "categories.csv"
    category_path.write_text(category_csv, encoding="utf-8")
    mixed_data = io.load_csv(category_path, dtype=object)

colour_column = mixed_data[:, :1]
colour_encoded = preprocessing.OneHotEncoder().fit_transform(colour_column)

print("CSV category column:", colour_column.ravel())
print("Encoded shape:", colour_encoded.shape)
print(colour_encoded)
np.testing.assert_array_equal(colour_encoded, [[0, 1], [1, 0], [0, 1]])

## 4. Sorting, searching and ranking

We sort a small array, find its largest values efficiently, select the third-smallest value using quickselect, search for an insertion position, and rank ties consistently.

In [ ]:
values = np.array([5, 2, 5, 1, 9, 3])
ordered = sort_search.stable_sort(values)
top_values, top_indices = sort_search.topk(values, 3)
third_smallest = sort_search.quickselect(values, 2)
insert_at, exists = sort_search.binary_search(ordered, 5)

print("Original:", values)
print("Sorted:", ordered)
print("Top 3 values:", top_values, "at indices", top_indices)
print("Third-smallest:", third_smallest)
print("Binary search for 5: index =", insert_at, ", found =", exists)
print("Average ranks:", rank.rank(values, method="average"))
print("Dense ranks:", rank.rank(values, method="dense"))
print("Percentiles [25, 50, 75]:", rank.percentile(values, [25, 50, 75]))

assert third_smallest == np.sort(values)[2]
assert exists and ordered[insert_at] == 5

### 4.1 Multi-key sorting and deterministic ties

`multikey_sort` orders rows using the first column as the primary key and the second to break ties. The `topk` operation also preserves a predictable choice when values tie at the cut-off.

In [ ]:
# First sort by column 0; break ties by column 1.
rows = np.array([[2, 9], [1, 5], [2, 3], [1, 7]])
sorted_rows, row_order = sort_search.multikey_sort(
    rows, keys=[0, 1], return_indices=True
)
print("Multi-key sorted rows:\n", sorted_rows)
print("Original row positions:", row_order)
np.testing.assert_array_equal(sorted_rows, [[1, 5], [1, 7], [2, 3], [2, 9]])

# At the cut-off, choose the earlier occurrence of a tied value.
tied_values, tied_indices = sort_search.topk([5, 9, 5, 5], 3)
print("Top 3 with ties:", tied_values, "indices:", tied_indices)
np.testing.assert_array_equal(tied_indices, [1, 0, 2])

## 5. Descriptive and streaming statistics

The toolkit supports scalar and axis-wise statistics, histograms, quantiles, and Welford-style streaming updates.

Missing CSV values are skipped for descriptive statistics rather than replaced with zeros. The preprocessing section uses imputation for a different purpose.

In [ ]:
print("Column means:", stats.mean(X, axis=0, skipna=True))
print("Column standard deviations:", stats.std(X, axis=0, skipna=True))
print("Median of first column:", stats.median(X_imputed[:, 0]))
print("Min / max:", stats.min(values), "/", stats.max(values))
print("Quantiles [0.25, 0.5, 0.75]:", stats.quantile(values, [0.25, 0.5, 0.75]))

counts, edges = stats.histogram(values, bins=3)
print("Histogram counts:", counts)
print("Histogram bin edges:", edges)

running = stats.RunningStats()
running.update(values[:3]).update(values[3:])
print("Streaming count / mean / variance:", running.count, running.mean, running.variance())
np.testing.assert_allclose(running.mean, np.mean(values))
np.testing.assert_allclose(running.variance(), np.var(values))
np.testing.assert_allclose(stats.mean(X, axis=0, skipna=True), [3.5, 38.0, 124.0])


## 6. Evaluation metrics

Use a simple example to show accuracy, precision, recall, F1, a confusion matrix, and regression mean-squared error.

In [ ]:
y_true = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])
y_pred = np.array([1, 1, 1, 0, 0, 1, 0, 0, 0, 0])

print("Accuracy:", metrics.accuracy(y_true, y_pred))
print("Precision:", metrics.precision(y_true, y_pred))
print("Recall:", metrics.recall(y_true, y_pred))
print("F1:", metrics.f1(y_true, y_pred))
print("Confusion matrix:\n", metrics.confusion_matrix(y_true, y_pred))
print("MSE example:", metrics.mse([1, 2, 3], [1, 2, 4]))

### 6.1 Verify metrics against known answers

Printed metrics can look plausible even when incorrect. These checks compare results to small values we can calculate by hand.

In [ ]:
# TP=3, FP=1, FN=2, TN=4; all four metrics have different values.
assert np.isclose(metrics.accuracy(y_true, y_pred), 7 / 10)
assert np.isclose(metrics.precision(y_true, y_pred), 3 / 4)
assert np.isclose(metrics.recall(y_true, y_pred), 3 / 5)
assert np.isclose(metrics.f1(y_true, y_pred), 2 / 3)
# Rows = actual class, columns = predicted class; labels ordered [0, 1].
np.testing.assert_array_equal(metrics.confusion_matrix(y_true, y_pred), [[4, 1], [2, 3]])
assert np.isclose(metrics.mse([1, 2, 3], [1, 2, 4]), 1 / 3)
print("Verified distinct metric scores, confusion matrix, and MSE")

## 7. Numerical stability and ROC / AUC

`utils.pairwise_distances` builds euclidean distance one feature at a time with `np.hypot`. The usual shortcut formula `||x||² + ||y||² - 2xy` loses the small difference between two large numbers, so it gives 0 for two points that are 1 apart. `utils.softmax` subtracts the maximum first, so large inputs do not overflow. The last lines score a binary classifier with `roc_curve` and `auc`.

In [ ]:
from numcompute import utils

far = np.array([[1e9], [1e9 + 1]])          # two points exactly 1 apart
squares = (far * far).sum(axis=1)
shortcut = np.sqrt(max(squares[0] + squares[1] - 2 * (far[0] @ far[1]), 0))
print("Shortcut formula gives:        ", shortcut)
print("NumCompute pairwise_distances: ", utils.pairwise_distances(far)[0, 1])
print("softmax([1000, 1000]):", utils.softmax([1000.0, 1000.0]))

roc_true = np.array([0, 1, 1, 0, 1, 0])
roc_scores = np.array([0.2, 0.9, 0.4, 0.3, 0.8, 0.6])
fpr, tpr, _ = metrics.roc_curve(roc_true, roc_scores)
print("AUC:", metrics.auc(fpr, tpr))

assert utils.pairwise_distances(far)[0, 1] == 1.0
np.testing.assert_allclose(utils.softmax([1000.0, 1000.0]), [0.5, 0.5])
assert np.isclose(metrics.auc(fpr, tpr), 8 / 9)

## 8. Finite-difference gradient and Jacobian

The numerical gradient approximates derivatives of a scalar-valued function. The Jacobian computes derivatives of each output of a vector-valued function. We compare both with known exact answers.

In [ ]:
point = np.array([1.0, 2.0])

def square_sum(v):
    return float(np.sum(v ** 2))

def vector_function(v):
    return np.array([v[0] ** 2 + v[1], v[0] - v[1] ** 2])

estimated_grad = optim.grad(square_sum, point, method="central")
estimated_jacobian = optim.jacobian(vector_function, point, method="central")

print("Gradient (estimated):", estimated_grad)
print("Gradient (exact):    ", 2 * point)
print("Jacobian (estimated):\n", estimated_jacobian)
print("Jacobian (exact):\n", np.array([[2.0, 1.0], [1.0, -4.0]]))

np.testing.assert_allclose(estimated_grad, 2 * point, atol=1e-6)
np.testing.assert_allclose(estimated_jacobian, [[2, 1], [1, -4]], atol=1e-6)

### 8.1 Forward versus central differences

For the same step size, the central-difference formula typically gives a more accurate numerical derivative for smooth functions. We can demonstrate this using the exact derivative of `sin(x)`.

In [ ]:
h = 1e-3
x0 = np.array([1.0])
exact = np.cos(1.0)
f_sin = lambda v: np.sin(v[0])
central = optim.grad(f_sin, x0, h=h, method="central")[0]
forward = optim.grad(f_sin, x0, h=h, method="forward")[0]
central_error = abs(central - exact)
forward_error = abs(forward - exact)
print("Central difference absolute error:", central_error)
print("Forward difference absolute error:", forward_error)
assert central_error < forward_error

## 9. Benchmark: Python loop versus NumPy

Both implementations calculate the **same sum of squares**. The helper checks that the answers match *before* measuring each function. It reports median timings and the speed-up ratio. Actual speed-ups depend on the computer and NumPy version.

In [ ]:
def loop_sum_of_squares(a):
    total = 0.0
    for number in a:
        total += number * number
    return total


def numpy_sum_of_squares(a):
    return float(np.sum(a * a))

sample = np.linspace(-3, 3, 100_000)
result = benchmarking.compare_loop_vs_vectorised(
    loop_sum_of_squares, numpy_sum_of_squares, sample,
    repeats=5, warmup=1,
)

print("| Implementation | Median time (ms) |")
print("|---|---:|")
print("| Python loop | {:.3f} |".format(result["loop"] * 1000))
print("| NumPy vectorised | {:.3f} |".format(result["vectorised"] * 1000))
print("Speed-up: {:.1f}x".format(result["speedup"]))
print("Environment:", benchmarking.environment_info())

assert result["loop"] > 0 and result["vectorised"] > 0

A single sum of squares is a simple test. This table adds two more routines from the toolkit. Each pair is checked to give the same answer before it is timed.

In [ ]:
def loop_distances(points):
    n = len(points)
    out = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            out[i][j] = sum((a - b) ** 2 for a, b in zip(points[i], points[j])) ** 0.5
    return np.array(out)


def loop_softmax(rows):
    out = []
    for row in rows:
        biggest = max(row)
        exps = [np.exp(v - biggest) for v in row]
        total = sum(exps)
        out.append([e / total for e in exps])
    return np.array(out)


rng = np.random.default_rng(0)
points = rng.normal(size=(150, 5))
logits = rng.normal(size=(2000, 10))

cases = [
    ("sum of squares (100,000 values)", loop_sum_of_squares, numpy_sum_of_squares, sample),
    ("pairwise distances (150 points)", lambda p: loop_distances(p.tolist()),
     utils.pairwise_distances, points),
    ("softmax (2000 rows)", lambda r: loop_softmax(r.tolist()), utils.softmax, logits),
]

# Plain-text aligned table, readable in notebook output without extra libraries.
header = "{:<35} {:>16} {:>17} {:>10}".format(
    "Routine", "Python loop (ms)", "NumCompute (ms)", "Speed-up"
)
print(header)
print("-" * len(header))
for name, loop_fn, vec_fn, data in cases:
    r = benchmarking.compare_loop_vs_vectorised(loop_fn, vec_fn, data, repeats=3, warmup=1)
    assert np.isclose(r["speedup"], r["loop"] / r["vectorised"])
    print("{:<35} {:>16.3f} {:>17.3f} {:>9.1f}x".format(
        name, r["loop"] * 1000, r["vectorised"] * 1000, r["speedup"]))

## 10. Wrap-up

- **I/O:** Loaded CSV text that had missing cells.
- **Preprocessing:** Filled NaNs, standardised and min-max scaled the data, and one-hot encoded categories.
- **Algorithms:** Sorting, top-k, quickselect, binary search and tie-aware ranks.
- **Statistics and metrics:** A histogram, quantiles, streaming variance, classification scores, ROC and AUC.
- **Stability:** Euclidean distance and softmax stay correct on very large values.
- **Optimisation:** Finite-difference gradient and Jacobian checked against the exact derivatives.
- **Performance:** Python loops compared with NumCompute on three routines.

Timings depend on the computer and the NumPy version, so they will be different on each machine.